<a href="https://colab.research.google.com/github/mayankrohilla-tech/Enterprise-Policy-Compliance-Assistant-LangChain-ChatBot-/blob/main/MayankRohilla_ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Enterprise Policy & Compliance Assistant

This notebook runs a full-featured backend + chat UI for ACME Industries' compliance assistant.

### What it does
| Capability | How |
|---|---|
| 📄 Document Q&A (RAG) | Ingest PDF/DOCX/TXT → FAISS vector store → cited answers |
| 🎥 YouTube Q&A | Transcript extraction → LLM synthesis |
| 🌐 Web Search | Tavily live search → synthesised answers with sources |
| 🧠 Multi-turn memory | Per-session history + cross-session summarisation |
| 📊 Telemetry | Tokens, latency, tool calls, errors — every request |

---
**Before running:** Add your API keys to the *Secrets* panel (🔑) in Colab:
- `OPENAI_API_KEY`
- `TAVILY_API_KEY`
- `NGROK_AUTHTOKEN` (free at ngrok.com)

Then **Runtime → Run all**.

In [1]:
# ── Cell 1: Install Dependencies ────────────────────────────────────────
# Remove old conflicting packages
!pip uninstall -y langchain langchain-core langchain-community langchain-openai

# Install latest compatible packages
# langchain        – Main LLM framework (chains, agents, prompts, memory, RAG)
# langchain-core   – Core abstractions: messages, runnables, output parsers
# langchain-community – Community integrations: vector DBs, loaders, tools
# langchain-openai – OpenAI integration (GPT models + embeddings)
# langgraph        – Stateful workflow orchestration and multi-agent routing
# langchain-text-splitters – Splits documents into chunks for RAG
# faiss-cpu        – Vector database for semantic similarity search
# openai           – Official OpenAI SDK
# pymupdf          – Fast PDF text extraction
# python-docx      – Read/process DOCX/Word files
# youtube-transcript-api – Fetch YouTube transcripts
# tavily-python    – Real-time web search API
# fastapi          – Backend API framework
# uvicorn          – ASGI server to run FastAPI
# nest-asyncio     – Fix async event-loop issues in Colab/Jupyter
# httpx            – Modern async HTTP client
# pyngrok          – Public tunnel for localhost/Colab apps
# tiktoken         – Token counting for OpenAI models

!pip install -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-openai \
    langgraph \
    langchain-text-splitters \
    faiss-cpu \
    openai \
    pymupdf \
    python-docx \
    youtube-transcript-api \
    tavily-python \
    fastapi \
    uvicorn \
    nest-asyncio \
    httpx \
    pyngrok \
    tiktoken                            # Token counting library for OpenAI models
print('✅ All dependencies installed')
print('⚠️  If this is the first run, go to Runtime → Restart session, then Run all again.')

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.3.1
Uninstalling langchain-core-1.3.1:
  Successfully uninstalled langchain-core-1.3.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ── Cell 2: Load API Keys from Colab Secrets ────────────────────────────
import os
from google.colab import userdata

try:
    os.environ['OPENAI_API_KEY']   = userdata.get('OPENAI_API_KEY')
    os.environ['TAVILY_API_KEY']   = userdata.get('TAVILY_API_KEY')
    os.environ['NGROK_AUTHTOKEN']  = userdata.get('NGROK_AUTHTOKEN')
    print('✅ API keys loaded')
except Exception as e:
    print(f'❌ Could not load secrets from Colab: {e}')
    print('   → Add OPENAI_API_KEY, TAVILY_API_KEY, NGROK_AUTHTOKEN to the Secrets panel (🔑)')
    raise

✅ API keys loaded


In [3]:
# ── Cell 3: Document Processor (PDF / DOCX / TXT) ───────────────────────
import fitz          # PyMuPDF
import docx
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_document(file_path: str) -> list[Document]:
    """Load PDF, DOCX, or TXT and return LangChain Documents."""
    path = Path(file_path)
    ext = path.suffix.lower()

    if ext == '.pdf':
        doc = fitz.open(file_path)
        pages = []
        for i, page in enumerate(doc):
            text = page.get_text()
            if text.strip():
                pages.append(Document(
                    page_content=text,
                    metadata={'source': path.name, 'page': i + 1}
                ))
        return pages

    elif ext == '.docx':
        d = docx.Document(file_path)
        full_text = '\n'.join(p.text for p in d.paragraphs if p.text.strip())
        return [Document(page_content=full_text, metadata={'source': path.name})]

    elif ext == '.txt':
        text = path.read_text(encoding='utf-8', errors='ignore')
        return [Document(page_content=text, metadata={'source': path.name})]

    else:
        raise ValueError(f'Unsupported file type: {ext}')


# Chunker
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)


def chunk_documents(docs: list[Document]) -> list[Document]:
    return splitter.split_documents(docs)


print('✅ Document processor ready')

✅ Document processor ready


In [4]:
# ── Cell 4: Vector Store (FAISS) ────────────────────────────────────────
import faiss
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Global corpus store + per-session ephemeral store
corpus_store: FAISS | None = None     #the persistent store. Holds documents loaded at startup (the sample policy files). Lives for the entire server lifetime.
session_store: FAISS | None = None    #the ephemeral store. Holds documents a user uploads mid-conversation. Intended to be reset between sessions.


def add_to_corpus(file_path: str) -> str:
    """Ingest a document into the persistent corpus vector store."""
    global corpus_store
    docs = chunk_documents(load_document(file_path))
    if corpus_store is None:
        corpus_store = FAISS.from_documents(docs, embeddings)
    else:
        corpus_store.add_documents(docs)
    return f'Ingested {len(docs)} chunks from {Path(file_path).name}'


def add_to_session(file_path: str) -> str:
    """Ingest a document into an ephemeral per-session store (mid-conversation upload)."""
    global session_store
    docs = chunk_documents(load_document(file_path))
    if session_store is None:
        session_store = FAISS.from_documents(docs, embeddings)
    else:
        session_store.add_documents(docs)
    return f'Session store: added {len(docs)} chunks from {Path(file_path).name}'


def reset_session_store():
    global session_store
    session_store = None


def retrieve(query: str, k: int = 5) -> list[Document]:
    """Search session store first, then fall back to corpus."""
    results = []
    if session_store:
        results += session_store.similarity_search(query, k=k)
    if corpus_store:
        results += corpus_store.similarity_search(query, k=k)
    # Deduplicate by content
    seen, unique = set(), []
    for d in results:
        key = d.page_content[:100]
        if key not in seen:
            seen.add(key)
            unique.append(d)
    return unique[:k]


print('✅ Vector store ready')

/tmp/ipykernel_1000/2912528430.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ Vector store ready


In [5]:
# ── Cell 5: YouTube Q&A Tool ─────────────────────────────────────────────
from youtube_transcript_api import YouTubeTranscriptApi
import re


def extract_video_id(url: str) -> str:
    patterns = [
        r'(?:v=|youtu\.be/)([A-Za-z0-9_-]{11})',
        r'embed/([A-Za-z0-9_-]{11})'
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    raise ValueError(f'Cannot extract video ID from URL: {url}')


def fetch_transcript(url: str) -> str:
    video_id = extract_video_id(url)
    try:
        # youtube-transcript-api >= 0.7.0
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)
        return ' '.join(t.text for t in transcript)
    except (TypeError, AttributeError):
        # youtube-transcript-api < 0.7.0 (legacy API)
        transcript = YouTubeTranscriptApi.get_transcript(video_id)
        return ' '.join(t['text'] for t in transcript)


print('✅ YouTube tool ready')

✅ YouTube tool ready


In [6]:
# ── Cell 6: Web Search Tool (Tavily) ────────────────────────────────────
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])


def web_search(query: str, max_results: int = 5) -> list[dict]:
    """Return list of {title, url, content} dicts."""
    try:
        response = tavily.search(query=query, max_results=max_results)
        return [
            {'title': r.get('title', ''), 'url': r.get('url', ''), 'content': r.get('content', '')}
            for r in response.get('results', [])
        ]
    except Exception as e:
        return [{'title': 'Search error', 'url': '', 'content': str(e)}]


print('✅ Web search tool ready')

✅ Web search tool ready


In [7]:
# ── Cell 7: Memory & Telemetry ───────────────────────────────────────────
import time       #Built-in module for time-related functions. Used here specifically for time.time() which returns the current Unix timestamp as a float — used to measure request latency by taking a snapshot before and after the LLM call.
import traceback  #dataclass — a decorator that auto-generates boilerplate methods (__init__, __repr__, __eq__) for a class based on its annotated fields. Saves writing a constructor manually.
from dataclasses import dataclass, field
#field — a helper used inside dataclasses for fields that need special default behaviour, particularly mutable defaults like lists (explained below).
#dataclass — a decorator that auto-generates boilerplate methods (__init__, __repr__, __eq__) for a class based on its annotated fields. Saves writing a constructor manually.
from collections import deque


@dataclass            #The @dataclass decorator inspects the class body, finds all annotated fields, and automatically generates an __init__ that accepts all of them as arguments with their defaults. Without this decorator you'd have to write def __init__(self, prompt_tokens=0, ...) manually.
class TelemetryRecord:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    latency_ms: float = 0.0
    tools_called: list[str] = field(default_factory=list)
    errors: list[str] = field(default_factory=list)


telemetry_log: list[TelemetryRecord] = []


class ConversationMemory:
    """Short-term in-memory history with summarisation after max_turns."""

    def __init__(self, max_turns: int = 10):
        self.max_turns = max_turns
        self.messages: list[dict] = []
        self.summary: str = ''

    def add(self, role: str, content: str):
        self.messages.append({'role': role, 'content': content})
        if len(self.messages) > self.max_turns * 2:
            self._compress()

    def _compress(self):
        """Summarise oldest half of messages to control context size."""
        cutoff = len(self.messages) // 2
        old = self.messages[:cutoff]
        self.messages = self.messages[cutoff:]
        # Build a simple extractive summary
        lines = [f"{m['role']}: {m['content'][:120]}" for m in old]
        self.summary = '[Earlier context]\n' + '\n'.join(lines)

    def get_history(self) -> list[dict]:
        if self.summary:
            return [{'role': 'system', 'content': self.summary}] + self.messages
        return self.messages

    def reset(self):
        self.messages = []
        self.summary = ''


memory = ConversationMemory()
print('✅ Memory & telemetry ready')

✅ Memory & telemetry ready


In [8]:
# ── Cell 8: Agent Orchestrator ───────────────────────────────────────────
import openai
import time
import json
import re

client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
MODEL = 'gpt-4o'

SYSTEM_PROMPT = """You are the ACME Industries Enterprise Policy & Compliance Assistant.
You help employees with internal policies, regulatory guidance, and general workplace queries.

When answering:
- Always cite document sources (filename + page) for RAG answers.
- For web search, cite URLs.
- For YouTube, mention the video title/URL.
- For conversational or general-knowledge questions, answer directly — do NOT invoke tools.
- Resolve follow-up questions using conversation history.
"""


def classify_intent(question: str, history: list[dict]) -> str:
    """
    Router: returns one of 'rag', 'youtube', 'web', 'general'.
    Uses a lightweight classification call.
    """
    routing_prompt = f"""
Classify the user query into exactly one category:
- rag        : questions about internal documents, policies, SOPs, or regulations in the corpus
- youtube    : query references a YouTube URL or asks about video content
- web        : needs current/live external information (news, recent amendments, prices)
- general    : greetings, general knowledge, conversational (no retrieval needed)

Query: {question}
Reply with exactly one word."""

    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': routing_prompt}],
        max_tokens=10,
        temperature=0,
    )
    return resp.choices[0].message.content.strip().lower()


def answer_rag(question: str, history: list[dict]) -> tuple[str, str, object]:
    """RAG: retrieve then generate. Returns (answer, 'rag')."""
    docs = retrieve(question)
    if not docs:
        context = 'No relevant documents found in the corpus.'
    else:
        chunks = []
        for d in docs:
            src = d.metadata.get('source', 'unknown')
            page = d.metadata.get('page', '')
            loc = f'{src}, p.{page}' if page else src
            chunks.append(f'[{loc}]\n{d.page_content}')
        context = '\n\n---\n\n'.join(chunks)

    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history + [
        {'role': 'user', 'content':
            f'Answer using only the documents below. Cite sources.\n\n{context}\n\nQuestion: {question}'}
    ]
    resp = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=800)
    return resp.choices[0].message.content, 'rag', resp.usage


def answer_youtube(question: str, url: str, history: list[dict]) -> tuple[str, str, object]:
    transcript = fetch_transcript(url)
    # Truncate if very long
    transcript = transcript[:6000]
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history + [
        {'role': 'user', 'content':
            f'Using this YouTube transcript (URL: {url}):\n\n{transcript}\n\nAnswer: {question}'}
    ]
    resp = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=800)
    return resp.choices[0].message.content, 'youtube', resp.usage


def answer_web(question: str, history: list[dict]) -> tuple[str, str, object]:
    results = web_search(question)
    context = '\n\n'.join(
        f'[{r["title"]}] ({r["url"]})\n{r["content"]}' for r in results
    )
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history + [
        {'role': 'user', 'content':
            f'Using these live web results, answer with source citations.\n\n{context}\n\nQuestion: {question}'}
    ]
    resp = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=800)
    return resp.choices[0].message.content, 'web', resp.usage


def answer_general(question: str, history: list[dict]) -> tuple[str, str, object]:
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history + [
        {'role': 'user', 'content': question}
    ]
    resp = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=600)
    return resp.choices[0].message.content, 'general', resp.usage


def chat(question: str, youtube_url: str | None = None) -> dict:
    """
    Main entry point. Returns:
      {'answer': str, 'tool': str, 'telemetry': TelemetryRecord}
    """
    t0 = time.time()
    tel = TelemetryRecord()
    history = memory.get_history()

    try:
        # Detect YouTube URL in question
        yt_pattern = r'https?://(?:www\.)?(?:youtube\.com|youtu\.be)/\S+'
        url_match = re.search(yt_pattern, question)
        if url_match:
            youtube_url = url_match.group(0)

        if youtube_url:
            intent = 'youtube'
        else:
            intent = classify_intent(question, history)

        tel.tools_called.append(intent)

        if intent == 'youtube' and youtube_url:
            answer, tool, usage = answer_youtube(question, youtube_url, history)
        elif intent == 'web':
            answer, tool, usage = answer_web(question, history)
        elif intent == 'rag':
            answer, tool, usage = answer_rag(question, history)
        else:
            answer, tool, usage = answer_general(question, history)

        tel.prompt_tokens = usage.prompt_tokens
        tel.completion_tokens = usage.completion_tokens

    except Exception as e:
        tel.errors.append(traceback.format_exc())
        # Graceful fallback
        try:
            answer, tool, usage = answer_general(question, history)
            tel.prompt_tokens = usage.prompt_tokens
            tel.completion_tokens = usage.completion_tokens
        except Exception as e2:
            answer = 'I encountered an error. Please try again.'
            tool = 'error'
            tel.errors.append(str(e2))

    tel.latency_ms = round((time.time() - t0) * 1000, 1)
    telemetry_log.append(tel)

    # Update memory
    memory.add('user', question)
    memory.add('assistant', answer)

    return {'answer': answer, 'tool': tool, 'telemetry': tel}


print('✅ Orchestrator ready')

✅ Orchestrator ready


In [9]:
# ── Cell 9: FastAPI Application ──────────────────────────────────────────
%%writefile app.py

import os, sys, asyncio, traceback
sys.path.insert(0, '/content')   # Colab working dir

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
import tempfile, shutil, json

# Import from notebook-defined globals (patched in at runtime)
# When running standalone, these are imported directly.
app = FastAPI(title='ACME Compliance Assistant', version='1.0.0')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'], allow_methods=['*'], allow_headers=['*']
)


class ChatRequest(BaseModel):
    question: str
    youtube_url: str | None = None


@app.get('/', response_class=HTMLResponse)
async def root():
    html = r'''
<!DOCTYPE html><html><head><title>ACME Compliance Assistant</title>
<meta charset="utf-8">
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: system-ui, sans-serif; background: #0d1b2a; color: #e8f4f8; min-height: 100vh; display: flex; flex-direction: column; align-items: center; padding: 24px; }
  h1 { font-size: 1.4rem; color: #02c39a; margin-bottom: 4px; }
  p.sub { font-size: 0.85rem; color: #64748b; margin-bottom: 18px; }
  #chat { width: 100%; max-width: 760px; flex: 1; overflow-y: auto; background: #132232; border-radius: 12px; padding: 16px; margin-bottom: 12px; min-height: 400px; }
  .msg { margin: 10px 0; padding: 10px 14px; border-radius: 8px; line-height: 1.6; max-width: 90%; }
  .user { background: #0b6e8f; margin-left: auto; text-align: right; }
  .bot  { background: #1e3a50; }
  .meta { font-size: 0.72rem; color: #64748b; margin-top: 4px; }
  #controls { width: 100%; max-width: 760px; display: flex; gap: 8px; }
  #q { flex: 1; padding: 10px 14px; border-radius: 8px; border: 1px solid #0b6e8f; background: #132232; color: #e8f4f8; font-size: 0.95rem; }
  button { padding: 10px 18px; border-radius: 8px; border: none; cursor: pointer; font-weight: 600; }
  #sendBtn { background: #0b6e8f; color: white; }
  #uploadBtn { background: #02c39a; color: #0d1b2a; }
  #fileInput { display: none; }
  #telPanel { width: 100%; max-width: 760px; margin-top: 12px; background: #132232; border-radius: 8px; padding: 10px 14px; font-size: 0.78rem; color: #64748b; }
</style></head><body>
<h1>🏛️ ACME Compliance Assistant</h1>
<p class="sub">Policy Q&A · YouTube · Web Search · Multi-turn Memory</p>
<div id="chat"></div>
<div id="controls">
  <input id="q" placeholder="Ask about GDPR, data retention, SEBI announcements, paste a YouTube URL..." />
  <button id="sendBtn">Send</button>
  <button id="uploadBtn">📎 Upload Doc</button>
  <input id="fileInput" type="file" accept=".pdf,.docx,.txt" />
</div>
<div id="telPanel">Telemetry will appear here after first message.</div>
<script>
const chat = document.getElementById('chat');
const tel  = document.getElementById('telPanel');
document.getElementById('q').addEventListener('keydown', e => { if (e.key === 'Enter') send(); });
document.getElementById('fileInput').addEventListener('change', function() { upload(this); });
document.getElementById('sendBtn').addEventListener('click', send);
document.getElementById('uploadBtn').addEventListener('click', () => document.getElementById('fileInput').click());

// ngrok requires this header to bypass its browser warning page
const HEADERS = {
  'Content-Type': 'application/json',
  'ngrok-skip-browser-warning': 'true'
};

function addMsg(text, cls, meta='') {
  const d = document.createElement('div');
  d.className = 'msg ' + cls;
  d.innerHTML = text.replace(/\n/g, '<br>');
  if (meta) { const m = document.createElement('div'); m.className='meta'; m.textContent=meta; d.appendChild(m); }
  chat.appendChild(d);
  chat.scrollTop = chat.scrollHeight;
  return d;
}

function setLoading(isLoading) {
  const btn = document.getElementById('sendBtn');
  const input = document.getElementById('q');
  btn.disabled = isLoading;
  btn.textContent = isLoading ? '⏳ Thinking...' : 'Send';
  input.disabled = isLoading;
}

async function send() {
  const q = document.getElementById('q').value.trim();
  if (!q) return;
  addMsg(q, 'user');
  document.getElementById('q').value = '';
  setLoading(true);
  const thinking = addMsg('⏳ Thinking...', 'bot');
  try {
    const res = await fetch(window.location.origin + '/chat', {
      method: 'POST',
      headers: HEADERS,
      body: JSON.stringify({question: q})
    });
    if (!res.ok) {
      const err = await res.text();
      throw new Error(`Server error ${res.status}: ${err}`);
    }
    const data = await res.json();
    const t = data.telemetry;
    thinking.remove();
    addMsg(data.answer, 'bot', `🔧 tool: ${data.tool}`);
    tel.textContent = `📊 Tokens: ${t.prompt_tokens} prompt + ${t.completion_tokens} completion | ⏱ ${t.latency_ms}ms | 🔧 ${t.tools_called.join(', ')} | ❌ Errors: ${t.errors.length}`;
  } catch(e) {
    thinking.remove();
    addMsg(`❌ Error: ${e.message}. Check that Cell 13 (server) is still running in Colab.`, 'bot');
    tel.textContent = `❌ Request failed: ${e.message}`;
  } finally {
    setLoading(false);
  }
}

async function upload(input) {
  const file = input.files[0];
  if (!file) return;
  const form = new FormData();
  form.append('file', file);
  const uploading = addMsg(`📎 Uploading ${file.name}...`, 'bot');
  try {
    const res = await fetch(window.location.origin + '/upload', {
      method: 'POST',
      headers: {'ngrok-skip-browser-warning': 'true'},
      body: form
    });
    if (!res.ok) {
      const err = await res.text();
      throw new Error(`Server error ${res.status}: ${err}`);
    }
    const data = await res.json();
    uploading.remove();
    addMsg(data.message, 'bot');
  } catch(e) {
    uploading.remove();
    addMsg(`❌ Upload failed: ${e.message}`, 'bot');
  }
  input.value = '';
}
</script></body></html>'''
    return html


@app.post('/chat')
async def chat_endpoint(req: ChatRequest):
    from notebook_bridge import chat   # patched at startup
    result = chat(req.question, req.youtube_url)
    return {
        'answer': result['answer'],
        'tool': result['tool'],
        'telemetry': {
            'prompt_tokens': result['telemetry'].prompt_tokens,
            'completion_tokens': result['telemetry'].completion_tokens,
            'latency_ms': result['telemetry'].latency_ms,
            'tools_called': result['telemetry'].tools_called,
            'errors': result['telemetry'].errors,
        }
    }


@app.post('/upload')
async def upload_endpoint(file: UploadFile = File(...)):
    from notebook_bridge import add_to_session
    suffix = '.' + file.filename.split('.')[-1]
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        shutil.copyfileobj(file.file, tmp)
        tmp_path = tmp.name
    try:
        msg = add_to_session(tmp_path)
        return {'message': f'✅ {msg}'}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get('/telemetry')
async def telemetry_endpoint():
    from notebook_bridge import telemetry_log
    return [{
        'prompt_tokens': t.prompt_tokens,
        'completion_tokens': t.completion_tokens,
        'latency_ms': t.latency_ms,
        'tools_called': t.tools_called,
        'errors': t.errors,
    } for t in telemetry_log]

Writing app.py


In [10]:
# ── Cell 10: Notebook Bridge (expose notebook globals to FastAPI) ─────────
import sys, types

# Create a module 'notebook_bridge' that FastAPI imports
bridge = types.ModuleType('notebook_bridge')
bridge.chat = chat
bridge.add_to_session = add_to_session
bridge.telemetry_log = telemetry_log
sys.modules['notebook_bridge'] = bridge

print('✅ Bridge module registered')

✅ Bridge module registered


In [11]:
# ── Cell 11: (Optional) Create Sample Policy Documents ───────────────────
sample_policies = {
    'data_retention_policy.txt': """ACME Industries — Data Retention Policy v2.3

1. HR Records: Employee records must be retained for 7 years after termination as per
   the Information Technology Act and DPDP Act 2023 guidelines.

2. Customer Data: Personal data collected for service delivery must not be retained
   beyond 3 years without explicit re-consent from the data principal.

3. Financial Records: All financial transaction logs must be retained for 8 years per
   the Companies Act 2013, Section 128.

4. Vendor Data: Third-party vendor contracts and associated data must be retained for
   5 years post contract expiry.

5. Data Deletion: Upon expiry of retention period, data must be securely destroyed
   using DoD 5220.22-M standard wipe or physical destruction for hardware.""",

    'vendor_data_sharing_policy.txt': """ACME Industries — Vendor Data Sharing Policy v1.1

1. Customer data may only be shared with vendors who have signed a Data Processing
   Agreement (DPA) and are listed in ACME's approved vendor register.

2. Before sharing any personal data with a vendor, obtain sign-off from:
   a) The Data Protection Officer (DPO)
   b) The relevant Business Unit Head

3. Vendors must not sub-process data without written consent from ACME.

4. Annual vendor audits are mandatory. Non-compliant vendors must be offboarded
   within 30 days of audit failure.

5. Cross-border data transfers require additional DPDP Act clearance and must route
   through ACME's approved data transfer mechanism.""",

    'gdpr_internal_notice.txt': """ACME Industries — GDPR Compliance Notice (EU Operations) v1.0

Applicability: All employees handling EU resident data.

Key Obligations:
- Lawful Basis: Every processing activity must have a documented lawful basis
  (consent, contract, legal obligation, legitimate interest).

- Data Subject Rights: Respond to access, erasure, and portability requests
  within 30 days. Escalate to DPO if unable to fulfill.

- Breach Notification: Report personal data breaches to the supervisory authority
  within 72 hours of discovery. Notify affected individuals without undue delay.

- Privacy by Design: New systems processing EU data must undergo DPIA before launch.

- EU Representative: ACME's EU representative is ACME GmbH, Frankfurt.""",
}

import tempfile, os
for filename, content in sample_policies.items():
    path = f'/tmp/{filename}'
    with open(path, 'w') as f:
        f.write(content)
    result = add_to_corpus(path)
    print(result)

print('\n✅ Sample corpus loaded into vector store')

Ingested 1 chunks from data_retention_policy.txt
Ingested 1 chunks from vendor_data_sharing_policy.txt
Ingested 1 chunks from gdpr_internal_notice.txt

✅ Sample corpus loaded into vector store


In [12]:
# ── Cell 12: Quick Local Test (no server needed) ─────────────────────────
print('=== Quick Test ===')

tests = [
    'Hi, what can you help me with?',
    'How long must we retain HR records?',
    'Can I share customer data with a new vendor?',
]

for q in tests:
    print(f'\n🙋 {q}')
    r = chat(q)
    print(f'🤖 {r["answer"][:300]}...' if len(r['answer']) > 300 else f'🤖 {r["answer"]}')
    t = r['telemetry']
    print(f'   📊 tool={r["tool"]} | tokens={t.prompt_tokens}+{t.completion_tokens} | {t.latency_ms}ms')

memory.reset()   # reset for the chat UI session

=== Quick Test ===

🙋 Hi, what can you help me with?
🤖 Hello! I can assist you with queries related to internal company policies, regulatory guidelines, and general workplace questions. If you have questions about ACME Industries’ procedures, need help understanding compliance requirements, or have a general workplace inquiry, feel free to ask!
   📊 tool=general | tokens=111+51 | 2730.2ms

🙋 How long must we retain HR records?
🤖 HR records must be retained for 7 years after termination, as per the Information Technology Act and DPDP Act 2023 guidelines (data_retention_policy.txt).
   📊 tool=rag | tokens=686+33 | 1897.7ms

🙋 Can I share customer data with a new vendor?
🤖 You can share customer data with a new vendor only after ensuring that the vendor has signed a Data Processing Agreement (DPA) and is listed in ACME's approved vendor register. Additionally, you must obtain sign-off from the Data Protection Officer (DPO) and the relevant Business Unit Head before sh...
   📊 tool=rag | t

In [13]:
# ── Cell 13: Launch FastAPI + ngrok Public URL ───────────────────────────
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf
import threading

nest_asyncio.apply()

# Set ngrok auth
conf.get_default().auth_token = os.environ['NGROK_AUTHTOKEN']

# Import the FastAPI app from file
import importlib.util, sys
spec = importlib.util.spec_from_file_location('app_module', '/content/app.py')
app_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app_module)
fastapi_app = app_module.app

PORT = 8000

def run_server():
    uvicorn.run(fastapi_app, host='0.0.0.0', port=PORT, log_level='warning')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
import time; time.sleep(2)  # wait for uvicorn to start

# Open ngrok tunnel
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url if hasattr(tunnel, 'public_url') else tunnel.url()

print('\n' + '='*60)
print(f'🚀  ACME Compliance Assistant is LIVE!')
print(f'🌐  Open this URL in your browser:')
print(f'    {public_url}')
print(f'📋  API docs:   {public_url}/docs')
print(f'📊  Telemetry: {public_url}/telemetry')
print('='*60)

# Display clickable link in Colab output
from IPython.display import display, HTML
display(HTML(f'''
<div style="padding:16px;background:#0d1b2a;border-radius:8px;font-family:system-ui">
  <h3 style="color:#02c39a;margin:0 0 8px">🏛️ ACME Compliance Assistant</h3>
  <a href="{public_url}" target="_blank"
     style="display:inline-block;padding:10px 20px;background:#0b6e8f;color:white;
            border-radius:6px;text-decoration:none;font-weight:600">
    Open Chat Interface →
  </a>
  <p style="color:#64748b;margin-top:8px;font-size:0.85rem">
    API docs at <a href="{public_url}/docs" target="_blank" style="color:#1ca3c4">{public_url}/docs</a>
  </p>
</div>
'''))


🚀  ACME Compliance Assistant is LIVE!
🌐  Open this URL in your browser:
    https://swinging-bunion-reissue.ngrok-free.dev
📋  API docs:   https://swinging-bunion-reissue.ngrok-free.dev/docs
📊  Telemetry: https://swinging-bunion-reissue.ngrok-free.dev/telemetry
